In [0]:
agro_df = spark.table("silver.agrofood.agrofood_normalized_fin").dropDuplicates()
display(agro_df)

In [0]:
kca_df = spark.table("silver.kca_info.kca_normalized_fin").dropDuplicates()
display(kca_df)

In [0]:
kca_df = kca_df.withColumnRenamed("업태명", "업태")

In [0]:
df = agro_df.unionByName(kca_df)
display(df)

In [0]:
from pyspark.sql.functions import when, col
df = df.withColumn("단위", when(col("단위") == "1리터", "1L").otherwise(col("단위")))

In [0]:
# 26.05.06 추가
# 가격 0원 또는 1원 null 처리
from pyspark.sql.functions import when

df = df.withColumn("가격", when((col("가격") == 0) | (col("가격") == 1), None).otherwise(col("가격")))

In [0]:
# 수집시간 형식 통일 (공백 -> 언더스코어)
from pyspark.sql.functions import regexp_replace, col

df = df.withColumn("수집시간", regexp_replace(col("수집시간"), " ", "_"))

In [0]:
# 딸기, 멜론, 수박, 참외 카테고리 변경 (채소류 -> 과일류)
from pyspark.sql.functions import when, col

df = df.withColumn("카테고리", when(col("재료명").isin("딸기", "멜론", "수박", "참외"), "과일류").otherwise(col("카테고리")))

In [0]:
from pyspark.sql.functions import when, col, regexp_replace

# 세부속성 변환 (추가)
df = df.withColumn(
    "세부속성",
    when(
        (col("재료명") == "계란") & col("세부속성").rlike("^특란[0-9]+"),
        "특란"
    ).otherwise(col("세부속성"))
)

# 단위 변환
df = df.withColumn(
    "단위",
    when(
        (col("재료명") == "계란") & col("단위").rlike("^[0-9]+구$"),
        regexp_replace(col("단위"), "구$", "개")
    ).otherwise(col("단위"))
)

# 단위_문자 변환
df = df.withColumn(
    "단위_문자",
    when(
        (col("재료명") == "계란") & (col("단위_문자") == "구"),
        "개"
    ).otherwise(col("단위_문자"))
)

In [0]:
display(df)

In [0]:
# from pyspark.sql.functions import regexp_replace

# df = df.withColumn("가격", regexp_replace(col("가격"), "[^0-9]", "").cast("int")) \
#        .withColumn("단위_수치", regexp_replace(col("단위_수치"), "[^0-9]", "").cast("int"))
# display(df)

In [0]:
# from pyspark.sql.functions import to_date

# df = df.withColumn("날짜", to_date(col("날짜"), "yyyyMMdd"))
# display(df)

In [0]:
# 테이블 저장
df.dropDuplicates() \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.ingredient.ingredient")

In [0]:
# csv 저장
df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/silver/ingredient/ingredient")

In [0]:
%skip
display(
    df.filter(col("출처") == "agrofood").limit(100)
    .unionByName(df.filter(col("출처") == "kca").limit(100))
)